# 02 · Data audit, premise check, tuning (run once, T4)
1. What is in the data: isolates and % resistant per pair and year; whether acquisition dates exist
   (if not, the 2017 pool is split into 4 fixed pseudo-batches instead of calendar quarters).
2. **Premise check** - does a 2015-16 model get worse over time, and how much does full retraining recover on 2018?
   If full retraining barely helps, the temporal question (RQ1) is weak and the cross-site question (RQ4) should lead.
3. Hyper-parameter tuning on the training years only (2015-16), saved to `config/best_params.json`.

In [ ]:
# ---- setup: identical in every notebook ----
FOLDER = "AMR_AL"          # name of the shared folder (or its shortcut) in My Drive

from google.colab import drive
drive.mount("/content/drive")
import os, sys, subprocess
ROOT = f"/content/drive/MyDrive/{FOLDER}"
if not os.path.isdir(os.path.join(ROOT, "amr_al")):
    raise SystemExit(f"{ROOT}/amr_al not found.\nIn Google Drive open 'Shared with me', right-click {FOLDER} -> "
                     "Organize -> Add shortcut -> My Drive, then run this cell again.")
try:
    import xgboost
    if int(xgboost.__version__.split(".")[0]) < 2:
        raise ImportError
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xgboost>=2.0"], check=True)
os.environ["AMR_AL_ROOT"] = ROOT
sys.path.insert(0, ROOT)
for m in [k for k in sys.modules if k.startswith("amr_al")]:
    del sys.modules[m]                      # always load the latest code from Drive
from amr_al import config, paths, data, models, runner, audit, analysis, driams_fetch
paths.ensure_dirs()
print("shared folder:", ROOT)
print("compute device for XGBoost:", models.device())

In [ ]:
data.stage_to_local(["DRIAMS-A"])      # copy to the fast Colab disk (1-2 min)

## 1 · What is in the data

In [ ]:
sites, pairs = audit.summarize()
display(sites)
display(pairs.pivot_table(index=["pair", "site"], columns="year", values=["n", "pct_R"]))

## 2 · Tuning (training years only; roughly 5-15 min per pair on a T4)

In [ ]:
import pandas as pd
tuned = pd.concat([audit.tune(pair) for pair in config.PAIRS])
display(tuned.sort_values(["pair", "auroc"], ascending=[True, False]).groupby("pair").head(3))

## 3 · Premise check

In [ ]:
prem = pd.concat([audit.premise_check(pair) for pair in config.PAIRS])
prem.to_csv(os.path.join(paths.TABLES, "premise_check.csv"), index=False)
display(prem[["pair", "model", "eval_on", "auroc", "auprc", "vme", "me", "n", "n_R", "rounds_mode"]])
for pair, g in prem.groupby("pair"):
    none = g[(g.model == "no_retrain") & (g.eval_on == config.TEST_YEAR)].auroc.item()
    full = g[g.model == "retrain_with_all_2017"].auroc.item()
    cv = g[g.model == "in_period_5fold_cv"].auroc.item()
    print(f"{pair}: in-period {cv:.3f} | 2018 without retraining {none:.3f} | 2018 with full retraining {full:.3f} "
          f"| gap retraining can close = {full - none:+.3f}")

**Record:** both tables and the three `gap` lines. A gap under about 0.02 AUROC means retraining
barely matters for that pair - bring that to the supervisors before running E1 at full size.